# Mizanak — merged data and actual model results

Read the exported cohort and reports. To regenerate them, run download_data.py, nhanes_ai.train and export_dataset.py from the project folder. No model fitting takes place in this notebook.


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data/processed/NHANES_MIZAN.csv").exists():
    ROOT = ROOT.parent
cohort = pd.read_csv(ROOT / "data/processed/NHANES_MIZAN.csv")
audit = json.loads((ROOT / "reports/cohort.json").read_text(encoding="utf-8"))
assert len(cohort) == audit["remaining_records"]
print(f"Participants: {len(cohort):,}; columns: {len(cohort.columns)}")
cohort.groupby(["source_era", "age_group"]).size().unstack(fill_value=0)


## Data partition and known outcomes

Unknown outcomes remain missing and are excluded separately for each target. ID, survey design and partitions never enter predictors.


In [ ]:
cohort.groupby(["partition", "cycle"]).size().to_frame("participants")


In [ ]:
pd.DataFrame(audit["targets"]).T


## Validation comparison and held-out age results

Four candidates per disease. Selection uses validation age-macro average precision; test results do not tune the model or thresholds.


In [ ]:
pd.read_csv(ROOT / "reports/model_comparison.csv").sort_values(["disease", "age_macro_ap"], ascending=[True, False])


In [ ]:
pd.read_csv(ROOT / "reports/age_validation.csv")


## Interpretation

Experimental cross-sectional associations, not future disease risk or clinical diagnoses. Ages60–79 come only from historical data. Some rare outcomes perform poorly. Home-scale and population transfer remain unvalidated. See reports/MODEL_CARD.md for all results.
